aim is to select categories in information technology, or title includes titles that usually tech sectors need to hire for. and from that filter, i find what is the median and average salary, as well as the min and max range for each role.

In [25]:
# Importing packages, Pandas, Numpy, RegEx, json
import pandas as pd
import numpy as np
import re


In [26]:
df = pd.read_csv('SGJobData.csv')

df.describe(include='all')

,categories,employmentTypes,metadata_expiryDate,metadata_isPostedOnBehalf,metadata_jobPostId,metadata_newPostingDate,metadata_originalPostingDate,metadata_repostCount,metadata_totalNumberJobApplication,metadata_totalNumberOfView,...,occupationId,positionLevels,postedCompany_name,salary_maximum,salary_minimum,salary_type,status_id,status_jobStatus,title,average_salary
count,1044597,1044597,1044597,1048585,1044597,1044597,1044597,1.048585e+06,1.048585e+06,1.048585e+06,...,0.0,1044597,1044597,1.048585e+06,1.048585e+06,1044597,1048585.0,1044597,1044597,1.048585e+06
unique,21125,8,453,2,1044597,431,603,NaN,NaN,NaN,...,NaN,9,53151,NaN,NaN,1,NaN,3,377084,NaN
top,"[{""id"":21,""category"":""Information Technology""}]",Permanent,2023-07-28,False,MCF-2023-0252866,2023-06-09,2023-07-14,NaN,NaN,NaN,...,NaN,Executive,THE SUPREME HR ADVISORY PTE. LTD.,NaN,NaN,Monthly,NaN,Open,SUPERVISOR,NaN
freq,92869,458139,4487,986717,1,4508,4029,NaN,NaN,NaN,...,NaN,253701,61638,NaN,NaN,1044597,NaN,902614,8331,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.472327e-02,2.136571e+00,2.674536e+01,...,NaN,NaN,NaN,5.723578e+03,3.815312e+03,NaN,0.0,NaN,NaN,4.769445e+03
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.822675e-01,1.062612e+01,8.262001e+01,...,NaN,NaN,NaN,5.018387e+04,3.172182e+03,NaN,0.0,NaN,NaN,2.547809e+04
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,0.000000e+00,...,NaN,NaN,NaN,0.000000e+00,0.000000e+00,NaN,0.0,NaN,NaN,0.000000e+00
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,1.000000e+00,...,NaN,NaN,NaN,3.300000e+03,2.500000e+03,NaN,0.0,NaN,NaN,2.900000e+03
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,0.000000e+00,4.000000e+00,...,NaN,NaN,NaN,4.500000e+03,3.000000e+03,NaN,0.0,NaN,NaN,3.800000e+03
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,1.000000e+00,1.700000e+01,...,NaN,NaN,NaN,6.500000e+03,4.500000e+03,NaN,0.0,NaN,NaN,5.500000e+03


# Removing Missing Values (NaN)

In [27]:
# Checking missing values

missing_values = df.isna().sum()

print(missing_values[missing_values > 0].sort_values(ascending=False))

occupationId                    1048585
categories                         3988
employmentTypes                    3988
metadata_expiryDate                3988
metadata_jobPostId                 3988
metadata_newPostingDate            3988
metadata_originalPostingDate       3988
positionLevels                     3988
postedCompany_name                 3988
salary_type                        3988
status_jobStatus                   3988
title                              3988
dtype: int64


In [28]:
# Entire row of occupationId is empty. df.dropna() will drop every single row.

# We use jobs id column which is unique, except for the 3988 empty rows, to drop the empty jobs id rows.
df_clean = df.dropna(subset=['metadata_jobPostId']).copy()

# Drop empty occupationId column
df_clean = df_clean.drop(columns=['occupationId'])


# Parsing Date type

In [29]:
# Parsing the date columns - convert date object to datetime 

date_columns = ['metadata_expiryDate', 'metadata_newPostingDate', 'metadata_originalPostingDate']

for c in date_columns:
    df_clean[c] = pd.to_datetime(df_clean[c], errors='coerce')

In [30]:
print('Check for unparseable dates:', {c: df_clean[c].isna().sum() for c in date_columns})
print('Data date range:', df_clean['metadata_originalPostingDate'].min().date(), '->',
                     df_clean['metadata_originalPostingDate'].max().date())

Check for unparseable dates: {'metadata_expiryDate': 0, 'metadata_newPostingDate': 0, 'metadata_originalPostingDate': 0}
Data date range: 2022-10-03 -> 2024-05-29


# Cleaning 'title' Column

In [31]:
REGEX_RULES = [
    ('recruiter_code', r'^\s*\d{3,6}\s*[-:]\s*', ' '),          # recruiter codes at the start
    ('hashtags', r'#\w+', ' '),                                 # hashtags
    ('other_numbers', r'\b(?=[a-z0-9]*\d)[a-z0-9]{3,}\b', ' '), # any other numbers of 3 and above
    ('punction_emojis', r'[^a-z]', ' '),                        # punctuations, emojis
    ('collapse_ws', r'\s+', ' '),                               # collapse whitespace
]

lower_title = df_clean['title'].str.lower()                     # make all cell values lowercase

for name, pattern, replace in REGEX_RULES:                      # reassignment with regex, replace if pattern matches
    lower_title = lower_title.str.replace(pattern, replace, regex=True)

df_clean['title_clean'] = lower_title.str.strip()               # keep 'title' column, create new column 'title_clean' with normalized entry


# after cleaning, some rows become empty. if so, replace with original 'title'

df_clean['title_clean'] = df_clean['title_clean'].where(
    df_clean['title_clean'] != '', df_clean['title'].str.lower())

# Removing duplicated postings

In [32]:
# Duplicated postings, defined as new postings of similar job due to previous posting expiring.

key_parameters = ['title_clean', 'postedCompany_name', 'salary_minimum', 'salary_maximum']

# Sort by original posting date first, drop duplicates but keep first posting
df_clean = df_clean.sort_values('metadata_originalPostingDate').drop_duplicates(subset= key_parameters, keep='first')

print(df_clean.shape)


(629246, 22)


# Dealing with Salary Outliers

In [33]:
sal = ['salary_minimum', 'salary_maximum']

# Replace any empty salary with NaN
df_clean[sal] = df_clean[sal].replace(0, np.nan)

# Rebuild without NaN
df_clean['average_salary'] = df_clean[sal].mean(axis=1)

# Set monthly pay range to remove typos
df_clean_salary = df_clean[df_clean['average_salary'].between(800, 100000)].copy()